In [16]:
from pathlib import Path

from score.score import run
from score.data import load_metrics

from plot.metric_network_mean import plot_raw_network_metric
from plot.utils import load_ranks
from plot.rank_map import make_map
from plot.score_trends import make_trends
from plot.trend_fits import make_trend_fits
from plot.trend_map import make_trend_map
from plot.igc20_map import make_agreement_map
from plot.lat_ranks import (
    make_lat_ranks,
    make_lat_trends,
)
from plot.station_map import make_station_map
from config import load_config, Variant

import pandas as pd
IGS_CSV = "data/igs_stations.csv"

In [17]:
# Figure 1
# Load stations from IGS stations csv
stations = pd.read_csv(IGS_CSV)
stations["station"] = stations["Site Name"].astype(str).str[:4]
make_station_map(
    stations_file="data/stations.txt",
    igs_csv=IGS_CSV,
    plots_dir=None,
    include_titles=False
)

In [18]:

# PPP-augmented metrics
ppp_dir = Path("./results") / "ppp" / Variant.topsis.name
ppp_config = Path("scenarios/ppp.yaml")
ppp_config = load_config(ppp_config)

# Figure 3
raw_df = load_metrics(ppp_config)
plot_raw_network_metric(
    raw_df,
    "phase_wrms"
)

In [19]:

# Figure 4
ranks, metric_cols = load_ranks(ppp_config, Variant.topsis, stations)
make_agreement_map(
    ranks,
    metric_cols,
    ppp_dir,
    stations,
    title="PPP-augmented ranks in IGc20 clusters"
)

In [20]:

# Preprocessed metrics
preprocessed_dir = Path("./results") / "preprocessed" / Variant.topsis.name
preprocessed_config = Path("scenarios/preprocessed.yaml")
preprocessed_config = load_config(preprocessed_config)
preproc_ranks, preproc_metric_cols = load_ranks(preprocessed_config, Variant.topsis, stations)

# Figure 5
make_agreement_map(
    preproc_ranks,
    preproc_metric_cols,
    preprocessed_dir,
    stations,
    title="Preprocessed metrics - ranks in IGc20 clusters"
)

In [21]:
# Table 4
from IPython.display import HTML
df = ranks[["station", "score", "rank"]]
df = pd.concat([df.head(5), df.tail(5)])
HTML(df.to_html(index=False))


station,score,rank
BRUX,0.813708,1
HERS,0.805399,2
STJ3,0.805399,3
VNDP,0.801998,4
RIO2,0.792373,5
NLIB,0.540873,142
CIBG,0.521495,143
UTQI,0.486928,144
SALU,0.478185,145
POHN,0.471075,146


In [22]:
# Figure 6
make_map(
    ranks,
    metric_cols,
    title="Station Ranks - PPP-augmented",
)

In [23]:
# Figure 7
make_lat_ranks(
    ranks,
    title="PPP-augmented metrics - Average rank by geographic latitude"
)

In [24]:

# Run PPP for each day over the timeseries
ppp_windowed_dir = Path("./results") / "ppp_windowed" / Variant.topsis.name
ppp_windowed_config = Path("scenarios/ppp_windowed.yaml")
ppp_windowed_config = load_config(ppp_windowed_config)

run(ppp_windowed_config)

In [25]:
# Table 5
ppp_windowed_ranks, ppp_windowed_metric_cols = load_ranks(ppp_windowed_config, Variant.topsis, stations)
_, fits = make_trend_fits(ppp_windowed_ranks, ppp_windowed_dir, ppp_windowed_config.name, ppp_windowed_config.weights)

n=5
station_rank = ranks[["station", "rank"]].rename(columns={"rank": "rank"})
table = pd.concat([fits.head(n), fits.tail(n)])[["station", "slope_per_year"]]
table["direction"] = table["slope_per_year"].apply(lambda x: "Increasing" if x > 0 else "Decreasing")
table["slope_per_year"] = table["slope_per_year"].round(3)
table = table.merge(station_rank, on="station")[["rank", "station", "slope_per_year", "direction"]]
table

,rank,station,slope_per_year,direction
0,122,AIRA,0.031,Increasing
1,23,POL2,0.023,Increasing
2,33,PALM,0.021,Increasing
3,75,POLV,0.018,Increasing
4,56,BSHM,0.016,Increasing
5,53,DAV1,-0.025,Decreasing
6,129,SCTB,-0.025,Decreasing
7,76,LCK4,-0.025,Decreasing
8,65,CAS1,-0.026,Decreasing
9,63,ISPA,-0.028,Decreasing


In [26]:
# Figure 8
make_trend_map(
    fits,
    ranks,
    title=("Score trend per station")
)

In [27]:
# Figure 9
make_lat_trends(
    fits,
    ppp_windowed_ranks,
    title=("Mean score trend by geographic latitude (20 degree bins)"),
)

In [ ]:
def compare_metrics(ranks, metric_cols, station_a, station_b):
    subset = ranks[ranks["station"].isin([station_a, station_b])][["station"] + metric_cols]
    pivot = subset.set_index("station")[metric_cols].T.reset_index().rename(columns={"index":
    "metric"})
    weights_df = pd.DataFrame(ppp_config.weights.items(), columns=["metric", "weight"])
    table = pivot.merge(weights_df, on="metric").sort_values("weight", ascending=False)
    
    return table

# Table 6
table = compare_metrics(ranks, metric_cols, "DARW", "TOW2")
table[["metric", "weight", "DARW", "TOW2"]].round(3).reset_index(True)

,metric,weight,DARW,TOW2
0,phase_wrms,0.225,0.325,0.609
1,amb_resets,0.175,0.857,0.946
2,code_wrms,0.100,0.422,0.698
3,h_conv,0.100,0.665,0.917
4,v_conv,0.100,0.821,0.868
5,satellite_gaps,0.075,0.352,0.907
6,uptime,0.075,0.838,0.939
7,cn0,0.050,0.283,0.443
8,mp1,0.017,0.295,0.278
9,mp2,0.017,0.348,0.271


In [29]:
# Figure 10
make_trends(
    ppp_windowed_ranks,
    metric_cols,
    title=("Station score over time - DARW"),
    stations=["DARW"],
)

In [30]:
# Table 7
table = compare_metrics(ranks, metric_cols, "RIO2", "FALK")
table[["metric", "weight", "RIO2", "FALK"]].round(3)

,metric,weight,RIO2,FALK
0,phase_wrms,0.225,0.762,0.615
1,amb_resets,0.175,0.955,0.935
2,code_wrms,0.100,0.770,0.531
3,h_conv,0.100,0.881,0.896
4,v_conv,0.100,0.903,0.879
5,satellite_gaps,0.075,0.574,0.317
6,uptime,0.075,0.969,0.993
7,cn0,0.050,0.502,0.218
8,mp1,0.017,0.391,0.289
9,mp2,0.017,0.329,0.259
